# Composite Figure — Leakage-Fixed Results
Builds a single 4-panel publication figure from JSON result files.
No embeddings or heavy data needed — runs in under 10 seconds.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/'

In [ ]:
import json, re
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 8,
    'axes.titlesize': 9,
    'axes.labelsize': 8,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'figure.dpi': 300
})

def load_json(path):
    """Load JSON, replacing Infinity with a large finite number."""
    with open(path) as f:
        text = f.read()
    text = re.sub(r'\bInfinity\b', '1e308', text)
    return json.loads(text)

gnn    = load_json(DRIVE + 'noleak_gnn_results.json')
comp   = load_json(DRIVE + 'louvain_comparison_results.json')
corum  = load_json(DRIVE + 'noleak_corum_validation_results.json')
string = load_json(DRIVE + 'noleak_string_validation_results.json')

print('All results loaded.')
print(f"GNN: AUC-ROC={gnn['auc_roc']:.4f}, silhouette={gnn['silhouette']:.4f}")

In [ ]:
# ── Colours ───────────────────────────────────────────────────────────────────
C_GNN  = '#2196F3'
C_N2V  = '#FF9800'
C_LOU  = '#4CAF50'
C_RED  = '#E53935'
CLUSTER_COLORS = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
CLUSTER_LABELS = [
    'C1: Cytoplasmic signalling (47.0%)',
    'C2: Transcription & proteostasis (25.4%)',
    'C3: Cell cycle & DNA repair (16.7%)',
    'C4: Ribosome & OXPHOS (10.9%)'
]

# ── Layout ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(7.5, 7.2))
gs  = gridspec.GridSpec(2, 2, figure=fig,
                        height_ratios=[1, 1.15],
                        hspace=0.55, wspace=0.58)
ax_A = fig.add_subplot(gs[0, 0])
ax_B = fig.add_subplot(gs[0, 1])
ax_C = fig.add_subplot(gs[1, 0])
ax_D = fig.add_subplot(gs[1, 1])

for label, ax in zip(['A','B','C','D'], [ax_A, ax_B, ax_C, ax_D]):
    ax.text(-0.18, 1.10, label, transform=ax.transAxes,
            fontsize=11, fontweight='bold', va='top')

# ── Panel A: GNN performance metrics ─────────────────────────────────────────
metrics = ['AUC-ROC', 'AUC-PR', 'Silhouette']
values  = [gnn['auc_roc'], gnn['auc_pr'], gnn['silhouette']]
bars_a  = ax_A.bar(metrics, values, color=C_GNN, width=0.5, zorder=3)
ax_A.set_ylim(0, 1.15)
ax_A.axhline(0.5, color='grey', linestyle='--', linewidth=0.8, label='Random baseline')
ax_A.set_ylabel('Score')
ax_A.set_title('GNN Performance')
ax_A.yaxis.grid(True, alpha=0.3, zorder=0)
ax_A.set_axisbelow(True)
for bar, val in zip(bars_a, values):
    ax_A.text(bar.get_x() + bar.get_width()/2, val + 0.02,
              f'{val:.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
ax_A.legend(loc='lower right')

# ── Panel B: Baseline comparison ─────────────────────────────────────────────
methods   = ['GAT (ours)', 'node2vec', 'Louvain']
sil_vals  = [comp['GNN']['silhouette'],
             comp['node2vec']['silhouette'],
             comp['Louvain']['silhouette_full']]
corum_enr = [comp['GNN']['corum_enrichment'],
             comp['node2vec']['corum_enrichment'],
             comp['Louvain']['corum_enrichment']]
colors_b  = [C_GNN, C_N2V, C_LOU]
x = np.arange(len(methods))
w = 0.32

ax_B2 = ax_B.twinx()
b1 = ax_B.bar(x - w/2, sil_vals,  w, color=colors_b, alpha=0.9,  zorder=3)
b2 = ax_B2.bar(x + w/2, corum_enr, w, color=colors_b, alpha=0.45, hatch='//', zorder=3)
ax_B.axhline(0,   color='grey', linestyle='--', linewidth=0.8)
ax_B2.axhline(1.0, color=C_RED,  linestyle='--', linewidth=0.8)
ax_B.set_xticks(x)
ax_B.set_xticklabels(methods, fontsize=7)
ax_B.set_ylabel('Silhouette score', fontsize=7)
ax_B2.set_ylabel('CORUM enrichment (×)', fontsize=7)
ax_B.set_ylim(-0.25, 1.0)
ax_B2.set_ylim(0.4, 1.55)
ax_B.set_title('Baseline Comparison')
ax_B.yaxis.grid(True, alpha=0.3, zorder=0)
ax_B.set_axisbelow(True)
for bar, val in zip(b1, sil_vals):
    ax_B.text(bar.get_x() + bar.get_width()/2, max(val, 0) + 0.02,
              f'{val:.2f}', ha='center', va='bottom', fontsize=6, fontweight='bold')
for bar, val in zip(b2, corum_enr):
    ax_B2.text(bar.get_x() + bar.get_width()/2, val + 0.02,
               f'{val:.2f}×', ha='center', va='bottom', fontsize=6, fontweight='bold')
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_handles = [
    Patch(facecolor='grey', alpha=0.7, label='Silhouette (left axis)'),
    Patch(facecolor='grey', alpha=0.4, hatch='//', label='CORUM enrich. (right axis)'),
    Line2D([0],[0], color=C_RED, linestyle='--', linewidth=0.8, label='Random (1.0×)')
]
ax_B.legend(handles=legend_handles, loc='upper left',
            bbox_to_anchor=(0.01, 0.99), ncol=1, fontsize=5,
            frameon=True, borderpad=0.4, handlelength=1.0,
            handletextpad=0.4, labelspacing=0.3)

# ── Panel C: Module composition pie ──────────────────────────────────────────
sizes = [5211, 2818, 1854, 1205]
wedges, _, autotexts = ax_C.pie(
    sizes,
    labels=None,
    colors=CLUSTER_COLORS,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.72,
    wedgeprops={'linewidth': 0.8, 'edgecolor': 'white'}
)
for at in autotexts:
    at.set_fontsize(7)
    at.set_fontweight('bold')
    at.set_color('white')
ax_C.legend(wedges, CLUSTER_LABELS, loc='upper center',
            bbox_to_anchor=(0.5, -0.05), fontsize=6,
            frameon=True, borderpad=0.5, handlelength=1.2)
ax_C.set_title(f'Four Functional Modules\n(n=11,088 genes, sil={gnn["silhouette"]:.3f})')

# ── Panel D: Top CORUM complexes ──────────────────────────────────────────────
top      = corum['top_complexes'][:6]
names    = [t['complex'][:28] for t in top]
log_p    = [-np.log10(max(t['p_bonferroni'], 1e-300)) for t in top]
overlaps = [f"{t['overlap']}/{t['complex_size']}" for t in top]

y_pos  = np.arange(len(names))
bars_d = ax_D.barh(y_pos, log_p, color=C_GNN, alpha=0.85, height=0.55, zorder=3)
ax_D.set_yticks(y_pos)
ax_D.set_yticklabels(names, fontsize=6.5)
ax_D.invert_yaxis()
ax_D.set_xlabel('-log$_{10}$(p Bonferroni)')
ax_D.set_title('Top CORUM Complexes\n(Cluster 4, Bonferroni corrected)')
ax_D.axvline(-np.log10(0.05), color='grey', linestyle='--', linewidth=0.8)
ax_D.xaxis.grid(True, alpha=0.3, zorder=0)
ax_D.set_axisbelow(True)
for bar, ov in zip(bars_d, overlaps):
    ax_D.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
              ov, va='center', fontsize=6)

# ── Save ─────────────────────────────────────────────────────────────────────
plt.savefig(DRIVE + 'Figure_1_noleak.png', dpi=300, bbox_inches='tight')
plt.savefig(DRIVE + 'Figure_1_noleak.pdf', bbox_inches='tight')
plt.show()
print('Saved: Figure_1_noleak.png and Figure_1_noleak.pdf')